In [ ]:
from google.colab import drive
import geopandas as gpd
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import os

# Mount drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 1. SETUP INPUTS
outpath = "/content/drive/MyDrive/CCRI/ccri_repo/data/p2_vulnerability"

geojson_file_path = "/content/drive/MyDrive/CCRI/global_bnd_adm0.geojson"

# 2. DEFINE VARIABLES AND URLS

p2_variables = {
    "P2_At-least_basic_sanitation.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,WASH_HOUSEHOLDS,1.0/{iso3}.WS_PPL_S-ALB.._T._T",
    "P2_At-least_basic_drinking_water.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,WASH_HOUSEHOLDS,1.0/{iso3}.WS_PPL_W-ALB.WAT._T._T",
    "P2_Basic_hygiene.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,WASH_HOUSEHOLDS,1.0/{iso3}.WS_PPL_H-B.HYG._T._T",
    "P2_Under_five_mortality.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,CME,1.0/{iso3}.CME_MRY0T4._T._T?",
    "P2_Under_five_covered_by_social_protection.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,SOC_PROTECTION,1.0/{iso3}.SPP_CHLD_SOC_PROT.._T._T._T",
    "P2_Child_poverty.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,CHLD_PVTY,1.0/{iso3}.PV_CHLD_DPRV-S-L1-HS._T._T",
    "P2_Child_marriage.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,PT_CM,1.0/{iso3}.PT_F_20-24_MRD_U18._T.Y20T24._T._T._T._T._T._T._T._T",
    "P2_Child_labor.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,PT,1.0/{iso3}.PT_CHLD_5-17_LBR_ECON-HC._T.Y5T17._T._T._T._T",
    "P2_Learning_poverty.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,EDUCATION_FLS,1.0/{iso3}.ED_SE_LPV_PRIM._T",
    "P2_Lower_secondary_completion_rate.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,EDUCATION,1.0/{iso3}.ED_CR_L2._T.ISCED11_2._T._T.PCNT",
    "P2_Lower_secondary_out_of_school.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,EDUCATION,1.0/{iso3}.ED_ROFST_L2._T.ISCED11_2._T._T.PCNT",
    "P2_Child_Food_Poverty.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,NUTRITION,1.0/{iso3}.NT_CF_FG_0_T_2._T.M6T23._T._T._T._T",
    "P2_Stunting.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,NUTRITION,1.0/{iso3}.NT_ANT_HAZ_NE2_MOD._T.Y0T4._T._T._T._T",
    "P2_Skilled_birth_coverage.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,MNCH,1.0/{iso3}.MNCH_SAB._T.Y15T19._T._T._T._T.",
    "P2_DTP3_access.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,IMMUNISATION,1.0/{iso3}.IM_DTP3.DTP3.M12T23",
    "P2_DTP1_access.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,IMMUNISATION,1.0/{iso3}.IM_DTP1.DTP1.M12T23"
}

def fetch_country_data(iso3, api_url_template):
    """
    Fetch data for a given country using its ISO3 code.
    Returns the most recent observation as a dictionary, or None if data is missing.
    """
    api_url = api_url_template.format(iso3=iso3)
    try:
        response = requests.get(api_url)
        if response.status_code == 200:
            try:
                # Remove namespaces to simplify finding tags (optional but robust technique)
                # or just use standard parsing as you were doing
                root = ET.fromstring(response.content)
            except ET.ParseError as pe:
                print(f"XML parsing error for {iso3}: {pe}")
                return None

            data = []  # Initialize list OUTSIDE the series loop

            # ---------------------------------------------------------
            # CHANGE 1: Use findall() to get ALL Series elements, not just the first one
            # The XML structure separates years into different Series tags
            # ---------------------------------------------------------
            series_list = root.findall(".//Series")

            if not series_list:
                # Fallback: Sometimes namespaces mess up simple finds.
                # If series_list is empty, try searching with the specific namespace if needed,
                # but usually findall(".//Series") works if find(".//Series") worked.
                return None

            # ---------------------------------------------------------
            # CHANGE 2: Iterate through every Series found
            # ---------------------------------------------------------
            for series in series_list:
                obs_list = series.findall("Obs")
                if obs_list:
                    for obs in obs_list:
                        time_period_str = obs.get("TIME_PERIOD")
                        obs_value_str = obs.get("OBS_VALUE")
                        try:
                            time_period = int(time_period_str)
                            obs_value = float(obs_value_str)
                        except Exception as conv_err:
                            continue

                        data.append({
                            "iso3": iso3,
                            "time_period": time_period,
                            "obs_value": obs_value,
                            "data_source": obs.get("DATA_SOURCE") or series.get("DATA_SOURCE") # Sometimes source is on Series tag
                        })

            # ---------------------------------------------------------
            # CHANGE 3: Logic to find max is now after checking ALL series
            # ---------------------------------------------------------
            if data:
                most_recent = max(data, key=lambda x: x["time_period"])
                return most_recent
            else:
                return None
        else:
            print(f"Failed to fetch data for {iso3}. HTTP Status: {response.status_code}")
            return None
    except Exception as e:
        print(f"Error processing {iso3}: {e}")
        return None

# 4. LOAD GEODATA
# Load the GeoJSON file containing country boundaries and ISO3 codes
try:
    countries_gdf = gpd.read_file(geojson_file_path)
    print("GeoDataFrame loaded. Columns:", countries_gdf.columns)
    # Extract ISO3 codes (update the field name if it's not exactly 'iso3')
    iso3_codes = countries_gdf['iso3'].unique()
except Exception as e:
    print(f"Error loading GeoJSON: {e}")
    iso3_codes = []

# 5. MAIN ITERATION LOOP
if len(iso3_codes) > 0:
    for filename, api_template in p2_variables.items():
        print(f"\n==========================================")
        print(f"Processing P2 Variable: {filename}")
        print(f"Using API: {api_template}")
        print(f"==========================================")

        all_data = [] # Reset data list for this specific variable

        # Loop through ISO3 codes
        for iso3 in iso3_codes:
            # print(f"Processing {iso3}...") # Commented out to reduce noise, uncomment if debugging
            result = fetch_country_data(iso3, api_template)
            if result:
                all_data.append(result)

        # Export to CSV for this variable
        if all_data:
            df = pd.DataFrame(all_data)
            csv_file_path = os.path.join(outpath, filename)
            df.to_csv(csv_file_path, index=False)
            print(f"✔ Success! Exported {len(df)} records to {csv_file_path}")
        else:
            print(f"✘ Warning: No data fetched for {filename}")
else:
    print("No ISO3 codes found. Check your GeoJSON path.")